# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 | Edgar GUSCHING | B00822088  |
| 2 | Marius RANG | B00821794 |
| 3 | Victor HENZ | B00817998  |
| 4 | Ylias GUETARI | B00824832 |

**Group / repo name:** `aidams-lab1-gusching-rang-henz-guetari`  
**Submitter (one person):**  Edgar GUSCHING
**Repo URL:**  https://github.com/guschingedgar/aidams-lab1-gusching-rang-henz-guetari
**Streamlit Cloud URL (bonus):** https://aidams-lab1-gusching-rang-henz-guetari-citthygx7ko6rixwsgvrob.streamlit.app/

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.neighbors import BallTree

STEEL = "Nominal crude steel capacity (ttpa)"
IRON = "Nominal iron capacity (ttpa)"
AGE = "Plant age (years)"
EXPOSURE = "LitPop assets within 20 km (USD bn)"

In [2]:
file = "Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx"
plants = pd.read_excel(file, sheet_name="Plant data")
units = pd.read_excel(file, sheet_name="Plant capacities and status")

print(plants.columns.tolist())
plants.head()

['GEM plant ID', 'Plant name (English)', 'Plant name (other language)', 'Other plant names (English)', 'Other plant names (other language)', 'Owner', 'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status', 'Parent (English)', 'Parent GEM entity ID', 'Parent PermID', 'Location address', 'Location address (other language)', 'Municipality', 'Subnational unit', 'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy', 'GEM wiki page', 'Plant age', 'Announced date', 'Construction date', 'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date', 'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)', 'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products', 'Steel sector end users', 'Workforce size', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification', 'Main production equipment', 'Power source', 'Iron ore source', 'Met coal source']


,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Steel products,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,"billet, wire rod, angle, flat, bar, square bar...",unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,"billet, wire rod, rebar",building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,"wire rod, rebar, bar, billet, round bar, wire",unknown,4500,2025-10-06 00:00:00,unknown,no,EAF,unknown,NaN,unknown
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"billet, rebar","building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,"pipe, tube, flat","automotive, building and infrastructure, energ...",11000,2025-04-30 00:00:00,2025-12-04 00:00:00,no,DRI; EAF; BF; BOF,unknown,unknown,unknown


In [3]:
units.head()

,GEM plant ID,Plant name (English),Plant name (other language),Country/area,Main production equipment,Status,Start date,Nominal crude steel capacity (ttpa),Nominal BOF steel capacity (ttpa),Nominal EAF steel capacity (ttpa),Nominal IF steel capacity (ttpa),Other/unspecified steel capacity (ttpa),Nominal iron capacity (ttpa),Nominal BF capacity (ttpa),Nominal DRI capacity (ttpa),Other/unspecified iron capacity (ttpa)
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,Türkiye,EAF,operating,1983,1100,NaN,1100,NaN,NaN,NaN,NaN,NaN,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Namibia,EAF,construction,unknown,3000,NaN,3000,NaN,NaN,NaN,NaN,NaN,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,Russia,EAF,operating,2014,1600,NaN,1600,NaN,NaN,NaN,NaN,NaN,NaN
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,Bangladesh,EAF,operating,2015,1400,NaN,1400,NaN,NaN,NaN,NaN,NaN,NaN
4,P100000120284,Tianjin New Tiangang United Special Steel Co Ltd,天津天钢联合特钢有限公司，天津天钢联合钢铁有限公司,China,BF,retired,2009-06-24 00:00:00,NaN,NaN,NaN,NaN,NaN,2360,2360,NaN,NaN


The Excel file has two sheets we need. "Plant data" has one row per plant, with the name, owner, country, coordinates and age.

"Plant capacities and status" has one row per plant and status, with the capacity for each one. So one plant can show up several times there, and we need to add it up.

The age column is just called "Plant age" in the file. We rename it to "Plant age (years)" so it matches the lab instructions.

In [4]:
df = plants.rename(columns={"Plant age": AGE})

coordinates = df["Coordinates"].str.split(",", expand=True)
df["Latitude"] = pd.to_numeric(coordinates[0])
df["Longitude"] = pd.to_numeric(coordinates[1])

number_columns = [AGE, "Ferronickel capacity (ttpa)", "Sinter plant capacity (ttpa)",
                  "Coking plant capacity (ttpa)", "Pelletizing plant capacity (ttpa)"]
for column in number_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[["Plant name (English)", "Coordinates", "Latitude", "Longitude", AGE]].head()

,Plant name (English),Coordinates,Latitude,Longitude,Plant age (years)
0,Aba Iron and Steel Payas plant,"36.7474130, 36.2173300",36.747413,36.217330,43.0
1,Abba Steel Ohangwena steel plant,"-17.3978660, 15.8910220",-17.397866,15.891022,1.0
2,Abinsk Electric Steel Works,"44.8819380, 38.1275100",44.881938,38.127510,12.0
3,Abul Khair Steel Sitakunda plant,"22.4720080, 91.7348270",22.472008,91.734827,11.0
4,Acciaierie d'Italia Taranto steel plant,"40.5089930, 17.2075890",40.508993,17.207589,61.0


In [5]:
units[STEEL] = pd.to_numeric(units[STEEL], errors="coerce")
units[IRON] = pd.to_numeric(units[IRON], errors="coerce")

running = units[units["Status"].isin(["operating", "operating pre-retirement"])]
capacity = running.groupby("GEM plant ID")[[STEEL, IRON]].sum()

status_order = ["operating", "operating pre-retirement", "construction", "mothballed",
                "mothballed pre-retirement", "announced", "retired", "cancelled"]

def plant_status(statuses):
    for status in status_order:
        if status in statuses.values:
            return status
    return "unknown"

status = units.groupby("GEM plant ID")["Status"].apply(plant_status).rename("Plant status")
number_of_units = units.groupby("GEM plant ID").size().rename("Number of units")

df = df.merge(capacity, on="GEM plant ID", how="left")
df = df.merge(status, on="GEM plant ID", how="left")
df = df.merge(number_of_units, on="GEM plant ID", how="left")
df[STEEL] = df[STEEL].fillna(0)
df[IRON] = df[IRON].fillna(0)

df[["Plant name (English)", "Plant status", STEEL, IRON]].head()

,Plant name (English),Plant status,Nominal crude steel capacity (ttpa),Nominal iron capacity (ttpa)
0,Aba Iron and Steel Payas plant,operating,1100.0,0.0
1,Abba Steel Ohangwena steel plant,construction,0.0,0.0
2,Abinsk Electric Steel Works,operating,1600.0,0.0
3,Abul Khair Steel Sitakunda plant,operating,1400.0,0.0
4,Acciaierie d'Italia Taranto steel plant,operating pre-retirement,7800.0,4000.0


The coordinates come as text like "36.74, 36.21", so we split them into two number columns, Latitude and Longitude. We also turn age and the other capacity columns into numbers, so we can do maths on them.

For capacity, we only count units that are operating or operating pre-retirement. Announced, construction, mothballed, retired and cancelled units are left out, and a plant with nothing running gets 0.

Each plant also gets one status, which is its most active one (operating first, cancelled last). Keep in mind that capacity is what a plant can make in a year, not what it really makes.

---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [6]:
print("Number of plants:", len(df))
print("Unique plant IDs:", df["GEM plant ID"].nunique())

Number of plants: 1293
Unique plant IDs: 1293


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1293 entries, 0 to 1292
Data columns (total 50 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   GEM plant ID                         1293 non-null   str    
 1   Plant name (English)                 1293 non-null   str    
 2   Plant name (other language)          791 non-null    str    
 3   Other plant names (English)          742 non-null    str    
 4   Other plant names (other language)   335 non-null    str    
 5   Owner                                1293 non-null   str    
 6   Owner (other language)               578 non-null    str    
 7   Owner GEM entity ID                  1293 non-null   str    
 8   Owner PermID                         1293 non-null   object 
 9   SOE status                           212 non-null    str    
 10  Parent (English)                     1293 non-null   str    
 11  Parent GEM entity ID                 1293

In [8]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

Ferronickel capacity (ttpa)           1279
Pelletizing plant capacity (ttpa)     1178
Coking plant capacity (ttpa)          1156
Sinter plant capacity (ttpa)          1118
SOE status                            1081
Other plant names (other language)     958
Location address (other language)      794
Owner (other language)                 715
Other plant names (English)            551
Plant name (other language)            502
Plant age (years)                      168
Met coal source                        115
Iron ore source                         16
dtype: int64

In [9]:
unknown_text = (df == "unknown").sum()
unknown_text[unknown_text > 0].sort_values(ascending=False)

Pre-retirement announcement date    1257
Idled date                          1245
Retired date                        1185
Construction date                   1121
Iron ore source                     1089
Met coal source                     1085
Announced date                      1028
Power source                         807
ISO 50001                            671
Steel sector end users               594
Owner PermID                         551
ISO 14001                            385
Workforce size                       257
Location address                     143
Steel products                       133
Category steel product               131
Start date                           112
Municipality                         112
Parent PermID                         23
Subnational unit                      18
Owner                                  2
dtype: int64

There are 1,293 steel plants in the dataset. Each plant has its own ID and there are no duplicates, so one row really means one plant.

After cleaning, the table has 50 columns. Most of them are text (str), and the capacities, age and coordinates are numbers (float).

Yes, there are missing values. Age is missing for 168 plants, and sinter, coking, pelletizing and ferronickel capacity are empty for most plants. A lot of columns also say "unknown" instead of being blank, like the dates and the owner of 2 plants.

### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [10]:
capacity_columns = [STEEL, IRON, "Ferronickel capacity (ttpa)", "Sinter plant capacity (ttpa)",
                    "Coking plant capacity (ttpa)", "Pelletizing plant capacity (ttpa)"]
df[capacity_columns + ["Latitude", "Longitude", AGE]].describe()

,Nominal crude steel capacity (ttpa),Nominal iron capacity (ttpa),Ferronickel capacity (ttpa),Sinter plant capacity (ttpa),Coking plant capacity (ttpa),Pelletizing plant capacity (ttpa),Latitude,Longitude,Plant age (years)
count,1293.000000,1293.000000,14.000000,175.000000,137.000000,115.000000,1293.000000,1293.000000,1125.00000
mean,1723.675947,1284.054911,692.071429,4771.942857,1674.525547,3965.026087,30.107078,64.225291,38.78200
std,2652.317947,2640.585691,1020.085250,4140.451029,1346.584334,4918.962161,16.678049,66.403833,36.42716
min,0.000000,0.000000,54.000000,83.000000,60.000000,66.000000,-37.831379,-123.163599,0.00000
25%,0.000000,0.000000,112.500000,1675.000000,800.000000,1200.000000,23.504558,27.137563,16.00000
50%,850.000000,0.000000,275.000000,3750.000000,1200.000000,2500.000000,33.962272,87.295932,25.00000
75%,2040.000000,1400.000000,825.000000,6300.000000,2100.000000,5000.000000,39.976702,115.125838,55.00000
max,22999.000000,24750.000000,3400.000000,22063.000000,6500.000000,30000.000000,67.189096,174.728098,287.00000


In [11]:
plants_with_steel = df[df[STEEL] > 0]
all_capacity = df[capacity_columns].sum(axis=1)

print("Average crude steel capacity, all plants:", round(df[STEEL].mean()), "ttpa")
print("Average crude steel capacity, plants above 0:", round(plants_with_steel[STEEL].mean()), "ttpa")
print("Median crude steel capacity, plants above 0:", round(plants_with_steel[STEEL].median()), "ttpa")
print("Average of all capacity columns added together:", round(all_capacity.mean()), "ttpa")

Average crude steel capacity, all plants: 1724 ttpa
Average crude steel capacity, plants above 0: 2428 ttpa
Median crude steel capacity, plants above 0: 1300 ttpa
Average of all capacity columns added together: 4191 ttpa


In [12]:
print("Latitude from", df["Latitude"].min(), "to", df["Latitude"].max())
print("Longitude from", df["Longitude"].min(), "to", df["Longitude"].max())

Latitude from -37.831379 to 67.189096
Longitude from -123.163599 to 174.728098


In [13]:
wrong_sign = (df["Country/area"] == "United States") & (df["Longitude"] > 0)
df[wrong_sign][["Plant name (English)", "Country/area", "Subnational unit", "Coordinates"]]

,Plant name (English),Country/area,Subnational unit,Coordinates
524,Hyundai Steel Louisiana plant,United States,Louisiana,"30.5191000, 91.5209000"


In [14]:
df.loc[wrong_sign, "Longitude"] = -df.loc[wrong_sign, "Longitude"]
df[wrong_sign][["Plant name (English)", "Latitude", "Longitude"]]

,Plant name (English),Latitude,Longitude
524,Hyundai Steel Louisiana plant,30.5191,-91.5209


In [15]:
print(df[AGE].describe())
px.histogram(df, x=AGE, nbins=50, title="Distribution of plant age")

count    1125.00000
mean       38.78200
std        36.42716
min         0.00000
25%        16.00000
50%        25.00000
75%        55.00000
max       287.00000
Name: Plant age (years), dtype: float64


The average crude steel capacity is 1,724 ttpa over all plants. If we only keep plants above 0, it goes up to 2,428 ttpa, but the median is just 1,300 ttpa. That's because a few huge plants pull the average up, the biggest one is 22,999 ttpa.

If we add all the capacity columns together, a plant has 4,191 ttpa on average. Latitude goes from -37.8 (Australia) to 67.2 (Sweden), and longitude from -123.2 (US west coast) to 174.7 (New Zealand).

We also found that Hyundai Steel Louisiana had a positive longitude, which puts it in Asia, so we flipped the sign. For age, the median is 25 years and the mean is 39 years. Half of the plants are between 16 and 55 years old, and the oldest one is 287 years old.

### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [16]:
df["Region"].value_counts()

Region
Asia Pacific               765
Europe                     184
North America              113
Middle East                 90
Africa                      51
Eurasia                     47
Central & South America     43
Name: count, dtype: int64

In [17]:
country_counts = df["Country/area"].value_counts()
print(country_counts.head(10))
px.bar(country_counts.head(15), title="Top 15 countries by number of plants")

Country/area
China            458
India            113
United States     90
Iran              56
Japan             42
Russia            31
Türkiye           30
Vietnam           28
Brazil            25
Italy             24
Name: count, dtype: int64


In [18]:
owner_counts = df["Owner"].value_counts()
print("Number of owners:", len(owner_counts))
print("Owners with only one plant:", (owner_counts == 1).sum())
owner_counts.head(15)

Number of owners: 1069
Owners with only one plant: 962


Owner
Nucor Corp                              13
Cleveland-Cliffs Inc                    12
Nippon Steel Corp                       10
Commercial Metals Co                     8
Gerdau Ameristeel Corp                   8
Steel Authority of India Ltd             8
SteelAsia Manufacturing Corp             7
ArcelorMittal Brasil SA                  6
ArcelorMittal SA                         6
Liberty Steel Group                      6
Steel Dynamics Inc                       6
Tata Steel Ltd                           6
United States Steel Corp                 6
ArcelorMittal Nippon Steel India Ltd     5
JFE Steel Corp                           5
Name: count, dtype: int64

Asia Pacific has by far the most plants with 765, then Europe with 184 and North America with 113. By country, China is way ahead with 458 plants, then India with 113 and the United States with 90.

There are 1,069 different owner names, and 962 of them only own one plant. So most owners are small and the market is very spread out.

Nucor has the most plants (13), then Cleveland-Cliffs (12) and Nippon Steel (10). These are direct owners, so one big group can show up under several names.

### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?


In [19]:
total_capacity = df[STEEL].sum()
print("Total operating crude steel capacity:", round(total_capacity), "ttpa")
print("That is about", round(total_capacity / 1000), "million tonnes per year")

Total operating crude steel capacity: 2228713 ttpa
That is about 2229 million tonnes per year


In [20]:
capacity_by_owner = df.groupby("Owner")[STEEL].sum().sort_values(ascending=False)
capacity_by_owner.head(10)

Owner
Nippon Steel Corp               44423.0
POSCO Holdings Inc              41757.0
Angang Steel Co Ltd             30250.0
JFE Steel Corp                  28768.0
JSW Steel Ltd                   28359.0
Tata Steel Ltd                  26460.0
Hyundai Steel Co                24297.0
Cleveland-Cliffs Inc            23655.0
Steel Authority of India Ltd    21350.0
Baoshan Iron & Steel Co Ltd     19800.0
Name: Nominal crude steel capacity (ttpa), dtype: float64

In [21]:
capacity_by_region = df.groupby("Region")[STEEL].sum().sort_values(ascending=False)
print(capacity_by_region)
px.bar(capacity_by_region, title="Operating crude steel capacity by region")

Region
Asia Pacific               1557791.0
Europe                      266764.0
North America               149787.0
Eurasia                      92915.0
Middle East                  68038.0
Central & South America      55028.0
Africa                       38390.0
Name: Nominal crude steel capacity (ttpa), dtype: float64


In [22]:
capacity_by_country = df.groupby("Country/area")[STEEL].sum().sort_values(ascending=False)
share = (capacity_by_country / total_capacity * 100).round(1)
share.head(10)

Country/area
China            48.8
India             6.5
United States     5.0
Japan             4.8
Russia            3.8
South Korea       3.6
Türkiye           2.5
Germany           2.0
Brazil            1.9
Vietnam           1.8
Name: Nominal crude steel capacity (ttpa), dtype: float64

The total operating crude steel capacity is 2,228,713 ttpa, so about 2.2 billion tonnes per year. The owners with the most capacity are Nippon Steel, POSCO and Angang.

Asia Pacific has about 70% of all the capacity. China alone has 48.8%, so almost half the world, then India with 6.5% and the United States with 5.0%.

Africa and Central & South America have the least capacity. So steel making is really packed into a few places, mostly in Asia.

---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [23]:
fig = px.scatter_map(df, lat="Latitude", lon="Longitude", color="Region",
                     hover_name="Plant name (English)", hover_data=["Owner", "Country/area", STEEL],
                     zoom=1, height=850, map_style="carto-positron", title="Steel plant locations")
fig

Plants are mostly grouped in eastern China, Japan, India, Europe and the eastern US. You can clearly see the big cluster on the Chinese coast.

There are very few plants in Africa, South America and Central Asia. Those areas look almost empty next to Asia.

You can hover any point to see the plant name, owner and capacity. The map background needs internet to load.

### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [24]:
operating = df[df[STEEL] > 0]
fig = px.scatter_map(operating, lat="Latitude", lon="Longitude", size=STEEL, color="Owner",
                     hover_name="Plant name (English)", hover_data=["Country/area", STEEL, "Plant status"],
                     size_max=30, zoom=1, height=850, map_style="carto-positron",
                     title="Plants sized by crude steel capacity, coloured by owner")
fig.update_layout(showlegend=False)
fig

Bigger circles mean more capacity. The biggest ones are in China, Japan, South Korea and India.

There are more than 1,000 owners, so the colours repeat and we hid the legend because it was way too long. If you hover a point, you still see the real owner.

Plants with 0 operating capacity are not on this map, because a circle of size 0 can't be seen. They are still in all the tables.

### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [25]:
fig = px.density_map(df, lat="Latitude", lon="Longitude", radius=10,
                     zoom=1, height=850, map_style="carto-positron", title="Density of steel plants")
fig

The lab asks for density_mapbox, but it was removed in our Plotly version. density_map is the new name and it does the same thing.

The hottest area is clearly eastern China, where plants are packed really close together. Other dense areas are Japan, India, Europe and the eastern US.

This map is smoothed, so it shows where plants cluster, not an exact count. Zooming in or out changes how it looks.

---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.


### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [26]:
litpop_files = {
    "China": "litpop/LitPop_pc_300_arcsec_CHN_v1.hdf5",
    "India": "litpop/LitPop_pc_300_arcsec_IND_v1.hdf5",
    "Japan": "litpop/LitPop_pc_300_arcsec_JPN_v1.hdf5",
}

tables = []
for country, path in litpop_files.items():
    cells = pd.read_hdf(path, key="exposures")
    cells["Country/area"] = country
    tables.append(cells)

litpop = pd.concat(tables, ignore_index=True)
litpop = litpop.drop(columns="geometry")
print("Number of grid cells:", len(litpop))
litpop.head()

Number of grid cells: 182591


,value,latitude,longitude,region_id,impf_,Country/area
0,5.280440e+09,20.041667,110.208333,156,1,China
1,4.040559e+07,20.041667,110.625000,156,1,China
2,4.190224e+07,20.041667,110.708333,156,1,China
3,8.813872e+07,19.958333,109.541667,156,1,China
4,1.879947e+08,19.958333,109.625000,156,1,China


In [27]:
print(litpop.dtypes)
litpop.describe()

value           float64
latitude        float64
longitude       float64
region_id         int64
impf_             int64
Country/area        str
dtype: object


,value,latitude,longitude,region_id,impf_
count,1.825910e+05,182591.000000,182591.000000,182591.000000,182591.0
mean,3.817856e+08,33.585715,99.540705,207.031891,1.0
std,5.670982e+09,8.816306,17.568848,88.645570,0.0
min,0.000000e+00,6.875000,68.208333,156.000000,1.0
25%,5.023580e+04,27.041667,83.875000,156.000000,1.0
50%,1.295843e+06,33.875000,98.708333,156.000000,1.0
75%,1.401763e+07,40.375000,113.875000,156.000000,1.0
max,5.044057e+11,53.541667,145.791667,392.000000,1.0


In [28]:
latitudes = np.sort(litpop["latitude"].unique())
step = latitudes[1] - latitudes[0]
print("Distance between two grid rows:", round(step, 4), "degrees")
print("That is", round(step * 3600), "arcseconds, about", round(step * 111, 1), "km")

Distance between two grid rows: 0.0833 degrees
That is 300 arcseconds, about 9.2 km


value is the asset value of each grid cell in US dollars, for the year 2018. latitude and longitude give the centre of the cell, and region_id is the country code (156 China, 356 India, 392 Japan).

impf_ is an ID used by the CLIMADA tool, it's always 1 here so it's not useful for us. geometry is just a copy of the point, so we dropped it.

The grid is 300 arcseconds, so about 9 km between two cells. There's no population column in this sample, only asset value, and it only covers China, India and Japan.

### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [29]:
EARTH_RADIUS_KM = 6371

tree = BallTree(np.radians(litpop[["latitude", "longitude"]].values), metric="haversine")
plant_points = np.radians(df[["Latitude", "Longitude"]].values)

distance, index = tree.query(plant_points, k=1)
nearest_cell = litpop.iloc[index[:, 0]]

plants_exposure = df.copy()
plants_exposure["Distance to LitPop cell (km)"] = distance[:, 0] * EARTH_RADIUS_KM
plants_exposure["LitPop nearest cell value (USD)"] = nearest_cell["value"].values
plants_exposure["LitPop country"] = nearest_cell["Country/area"].values

In [30]:
cells_nearby = tree.query_radius(plant_points, r=20 / EARTH_RADIUS_KM)
values = litpop["value"].values
plants_exposure[EXPOSURE] = [values[cells].sum() / 1e9 for cells in cells_nearby]

In [31]:
good_match = (plants_exposure["Distance to LitPop cell (km)"] <= 20) & \
             (plants_exposure["LitPop country"] == plants_exposure["Country/area"])

litpop_columns = ["LitPop nearest cell value (USD)", EXPOSURE, "LitPop country"]
plants_exposure.loc[~good_match, litpop_columns] = np.nan

print("Plants matched to LitPop:", good_match.sum())
print(plants_exposure[good_match]["Country/area"].value_counts())
plants_exposure[good_match][["Plant name (English)", "Country/area", "Distance to LitPop cell (km)",
                             "LitPop nearest cell value (USD)", EXPOSURE]].head()

Plants matched to LitPop: 613
Country/area
China    458
India    113
Japan     42
Name: count, dtype: int64


,Plant name (English),Country/area,Distance to LitPop cell (km),LitPop nearest cell value (USD),LitPop assets within 20 km (USD bn)
12,Action Ispat and Power Jharsuguda steel plant,India,4.629910,6.450750e+08,5.805878
13,Adhunik Metaliks Kuanrmunda steel plant,India,3.434677,7.211772e+08,8.843561
21,Aichi Steel Chita plant (Tokai),Japan,5.250095,1.101869e+11,1237.334302
42,Angang Group Xinyang Iron and Steel Co Ltd,China,3.392302,3.822503e+08,0.638414
43,Angang Lianzhong Stainless Steel Co Ltd,China,3.671572,2.360539e+10,291.900767


In [32]:
plants_exposure[good_match]["Distance to LitPop cell (km)"].describe()

count    613.000000
mean       3.440857
std        1.616343
min        0.031360
25%        2.327039
50%        3.526021
75%        4.383673
max       18.271759
Name: Distance to LitPop cell (km), dtype: float64

We use a BallTree with the haversine distance, which measures real distance on the Earth. Each plant gets its nearest LitPop cell, and we also add up the value of all cells within 20 km around it.

We only keep a match if the cell is less than 20 km away and in the same country as the plant. That way a plant can't be matched to a cell across a border.

613 plants are matched: 458 in China, 113 in India and 42 in Japan. The other plants are outside the sample, so their LitPop values stay empty. The median distance to the nearest cell is 3.5 km, so the matches are pretty close.

### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [33]:
matched = plants_exposure[good_match].copy()
matched["Marker size"] = matched[STEEL] + 200

fig = px.scatter_map(matched, lat="Latitude", lon="Longitude", color=EXPOSURE, size="Marker size",
                     hover_name="Plant name (English)",
                     hover_data=["Owner", STEEL, EXPOSURE, "LitPop nearest cell value (USD)"],
                     range_color=(0, matched[EXPOSURE].quantile(0.95)), color_continuous_scale="Viridis",
                     size_max=25, zoom=2, center={"lat": 30, "lon": 105}, height=850,
                     map_style="carto-positron", title="Steel plants and asset value within 20 km (LitPop)")
fig

In [34]:
matched.groupby("Country/area")[EXPOSURE].median()

Country/area
China     12.880609
India      3.697783
Japan    112.164296
Name: LitPop assets within 20 km (USD bn), dtype: float64

In [35]:
big = matched[matched[STEEL] >= 10000]
small = matched[matched[STEEL] < 10000]
print("Median exposure, plants of 10 Mt or more:", round(big[EXPOSURE].median(), 1), "USD bn")
print("Median exposure, smaller plants:", round(small[EXPOSURE].median(), 1), "USD bn")

Median exposure, plants of 10 Mt or more: 35.7 USD bn
Median exposure, smaller plants: 11.2 USD bn


In [36]:
matched.sort_values(EXPOSURE, ascending=False)[["Plant name (English)", "Country/area", STEEL, EXPOSURE]].head(10)

,Plant name (English),Country/area,Nominal crude steel capacity (ttpa),LitPop assets within 20 km (USD bn)
758,Nakayama Steel Works Osaka,Japan,600.0,2751.209188
376,Godo Steel Osaka Works,Japan,500.0,2662.536169
757,Nakayama Steel Products Osaka plant,Japan,600.0,2540.160241
374,Godo Steel Funabashi Works,Japan,700.0,2537.905156
553,JFE East Japan Works (Keihin) steel plant,Japan,0.0,2479.218829
830,Osaka Steel Sakai Works,Japan,4650.0,2287.594026
657,Kyoei Steel Hirakata Division (Osaka),Japan,805.0,1494.338842
207,Chubu Steel Plate Nagoya plant,Japan,800.0,1425.227023
135,Baosteel Special Steel Co Ltd,China,0.0,1280.753311
21,Aichi Steel Chita plant (Tokai),Japan,1495.0,1237.334302


Japanese plants sit in very rich areas, with a median of about 112 USD bn within 20 km. Chinese plants are around 13 USD bn and Indian plants around 4 USD bn.

Plants of 10 Mt or more have a median of 36 USD bn around them, against 11 USD bn for smaller plants. So big plants tend to be near richer areas, but this doesn't tell us why.

The top 10 are almost all around Osaka, Tokyo and Nagoya. The colours stop at the 95th percentile, otherwise almost every point would look the same.

---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [37]:
with_owner = plants_exposure[plants_exposure["Owner"] != "unknown"]
groups = with_owner.groupby("Owner")

companies = pd.DataFrame()
companies["Number of plants"] = groups["GEM plant ID"].count()
companies["Total crude steel capacity (ttpa)"] = groups[STEEL].sum()
companies["Total iron capacity (ttpa)"] = groups[IRON].sum()
companies["Average plant age (years)"] = groups[AGE].mean()
companies["Average " + EXPOSURE] = groups[EXPOSURE].mean()
companies["Plants matched to LitPop"] = groups[EXPOSURE].count()
companies["Number of countries"] = groups["Country/area"].nunique()
companies["Number of regions"] = groups["Region"].nunique()
companies["Countries"] = groups["Country/area"].unique().apply(lambda names: ", ".join(sorted(names)))

companies = companies.sort_values("Total crude steel capacity (ttpa)", ascending=False)
companies.head(10)

,Number of plants,Total crude steel capacity (ttpa),Total iron capacity (ttpa),Average plant age (years),Average LitPop assets within 20 km (USD bn),Plants matched to LitPop,Number of countries,Number of regions,Countries
Owner,,,,,,,,,
Nippon Steel Corp,10,44423.0,42979.0,79.346000,160.027013,10,1,1,Japan
POSCO Holdings Inc,2,41757.0,41959.0,46.000000,NaN,0,1,1,South Korea
Angang Steel Co Ltd,3,30250.0,25610.0,22.006667,61.921445,3,1,1,China
JFE Steel Corp,5,28768.0,29651.0,62.494000,695.100823,5,1,1,Japan
JSW Steel Ltd,5,28359.0,29306.0,28.250000,3.221571,5,1,1,India
Tata Steel Ltd,6,26460.0,26040.0,52.600000,12.502733,5,2,1,"India, Singapore"
Hyundai Steel Co,4,24297.0,12440.0,46.500000,NaN,0,2,2,"South Korea, United States"
Cleveland-Cliffs Inc,12,23655.0,18360.0,107.333333,NaN,0,2,1,"Canada, United States"
Steel Authority of India Ltd,8,21350.0,23176.0,70.688333,19.336756,8,1,1,India


In [38]:
print("Number of companies:", len(companies))
print("Companies in more than one country:", (companies["Number of countries"] > 1).sum())

Number of companies: 1068
Companies in more than one country: 18


We group the plants by Owner and skip the 2 plants with an unknown owner. That gives us 1,068 companies, and only 18 of them have plants in more than one country.

Nippon Steel has the highest capacity, with 10 plants all in Japan. For each company we also get the total iron capacity, the average age and the list of countries.

The average LitPop exposure only uses plants in China, India and Japan. Companies with no plant there just have an empty value.

### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [39]:
companies["Latitude"] = groups["Latitude"].mean()
companies["Longitude"] = groups["Longitude"].mean()
companies[["Number of plants", "Countries", "Latitude", "Longitude"]].head(10)

,Number of plants,Countries,Latitude,Longitude
Owner,,,,
Nippon Steel Corp,10,Japan,35.297664,135.409815
POSCO Holdings Inc,2,South Korea,35.464698,128.571579
Angang Steel Co Ltd,3,China,40.995235,121.827942
JFE Steel Corp,5,Japan,35.658747,137.600034
JSW Steel Ltd,5,India,17.526863,78.327159
Tata Steel Ltd,6,"India, Singapore",19.936931,87.207839
Hyundai Steel Co,4,"South Korea, United States",35.250010,72.799805
Cleveland-Cliffs Inc,12,"Canada, United States",41.211779,-82.509657
Steel Authority of India Ltd,8,India,22.683159,85.717344


We use option 1, so each company is placed at the average latitude and longitude of its plants. It's the simplest option and it works for every company.

The downside is that the centre can land somewhere the company has no plant at all. For example, Hyundai Steel has plants in Korea and the US, so its centre ends up in South Asia.

For companies with plants in only one country, the centre stays close to their real plants, so it's fine for most of them.

### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [40]:
company_map = companies[companies["Average " + EXPOSURE].notna() & (companies["Total crude steel capacity (ttpa)"] > 0)]
company_map = company_map.reset_index()

fig = px.scatter_map(company_map, lat="Latitude", lon="Longitude", size="Total crude steel capacity (ttpa)",
                     color="Average " + EXPOSURE, hover_name="Owner",
                     hover_data=["Number of plants", "Total crude steel capacity (ttpa)", "Countries"],
                     range_color=(0, company_map["Average " + EXPOSURE].quantile(0.95)),
                     color_continuous_scale="Viridis", size_max=35, zoom=2, center={"lat": 30, "lon": 105},
                     height=850, map_style="carto-positron", title="Companies at the centre of their plants")
fig

Each circle is one company, and bigger means more capacity. The colour shows the average asset value within 20 km of its plants.

We only show companies with LitPop data and capacity above 0, so 410 companies. Japanese companies have the richest surroundings, with a median of about 734 USD bn.

Chinese companies are around 11 USD bn and Indian companies around 6 USD bn. The top ones are Nakayama Steel, Osaka Steel and Godo Steel, all near Osaka or Tokyo.

---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


app.py is in the same folder and follows this structure. The sidebar has filters for region, country, company, plant status and a capacity slider.

The main area shows the key numbers at the top, then the maps, charts and a data table. At the bottom there's a footer with the data sources and a few notes.

To open it, run streamlit run app.py in the terminal from this folder.

### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading


In [41]:
import os
os.makedirs("data/processed", exist_ok=True)

export_columns = ["GEM plant ID", "Plant name (English)", "Owner", "Parent (English)", "Country/area", "Region",
                  "Subnational unit", "Latitude", "Longitude", "Coordinate accuracy", "Plant status",
                  "Number of units", AGE, STEEL, IRON, "Ferronickel capacity (ttpa)",
                  "Sinter plant capacity (ttpa)", "Coking plant capacity (ttpa)",
                  "Pelletizing plant capacity (ttpa)", "Main production equipment", "Workforce size", "GEM wiki page"]
litpop_export = ["LitPop nearest cell value (USD)", EXPOSURE, "Distance to LitPop cell (km)", "LitPop country"]

df[export_columns].to_csv("data/processed/plants_clean.csv", index=False)
plants_exposure[export_columns + litpop_export].to_csv("data/processed/plants_with_litpop.csv", index=False)
companies.reset_index().to_csv("data/processed/companies.csv", index=False)
print(os.listdir("data/processed"))

['plants_with_litpop.csv', 'companies.csv', 'plants_clean.csv']


### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

Here we load the same CSV file the dashboard reads, and show what it displays when you first open it. By default the dashboard only keeps plants that are operating or operating pre-retirement.

The first cell gives the key numbers from the top of the dashboard. The charts below are the ones from the Exploration tab.

In [42]:
dashboard_plants = pd.read_csv("data/processed/plants_with_litpop.csv")
selection = dashboard_plants[dashboard_plants["Plant status"].isin(["operating", "operating pre-retirement"])]

print("Plants:", len(selection))
print("Operating capacity:", round(selection[STEEL].sum() / 1000), "Mt per year")
print("Companies:", selection["Owner"].nunique())
print("Countries:", selection["Country/area"].nunique())
print("Average plant age:", round(selection[AGE].mean()), "years")
print("Plants with LitPop data:", selection[EXPOSURE].notna().sum())

Plants: 970
Operating capacity: 2229 Mt per year


Companies: 830
Countries: 81
Average plant age: 39 years
Plants with LitPop data: 481


In [43]:
by_region = selection.groupby("Region")[STEEL].sum().sort_values()
px.bar(by_region, orientation="h", title="Capacity by region")

In [44]:
top_countries = selection.groupby("Country/area")[STEEL].sum().nlargest(15).sort_values()
px.bar(top_countries, orientation="h", title="Top 15 countries by capacity")

In [45]:
top_owners = selection.groupby("Owner")[STEEL].sum().nlargest(15).sort_values()
px.bar(top_owners, orientation="h", title="Top 15 owners by capacity")

In [46]:
px.histogram(selection, x=AGE, color="Region", nbins=50, title="Plant age distribution")

In [47]:
status_counts = dashboard_plants["Plant status"].value_counts()
px.bar(status_counts, title="Plants by status (all plants)")

With the default filters, the dashboard shows 970 plants, about 2,229 Mt per year of capacity, 830 companies and 81 countries. The average plant age is 39 years, and 481 of these plants have LitPop data.

The charts tell the same story as Part 2. Asia Pacific is far ahead on capacity, then Europe and North America, and the biggest owners are Nippon Steel, POSCO and Angang.

Most plants in the file are operating (935), but there are also 100 announced, 94 retired and 57 under construction. In the app, all of this updates when you change the filters, and there are also the maps, the company table and a data table you can download.

---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
